In [ ]:
import os
os.environ["PYTHONDONTWRITEBYTECODE"] = "0"

import logging
from dotenv import load_dotenv, find_dotenv

import pandas as pd

from langchain_openai import ChatOpenAI
from langchain_core.output_parsers import PydanticOutputParser
from langchain_core.rate_limiters import InMemoryRateLimiter

from core.flow.base import InferenceFlow
from core.flow.utils import (
    compute_routing_metrics,
    compute_stat_metrics,
    get_reranking_metrics,
    get_context_compression_metrics,

)
from core.data.datasets import SlmFlowSyntheticDataset
from core.router.base import SlmFlowScheduler
from core.router.policies import ThresholdRoutingPolicy
from core.router.features import (
    RerankingFeatureExtractor,
    ContextCompressionFeatureExtractor
)
from core.data.synthetic import (
    MessagesBuilder,
    DatasetDeclaration,
    AsyncDeclarativeDatasetGenerator
)
from core.io.models import (
    RagTaskOutput,
    SyntheticRowReranking,
    SyntheticRowCompression
)
from core.tasks.base import RagTask
from core.io.prompts import PROMPT_REGISTRY

import warnings
warnings.filterwarnings("ignore")

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s [%(name)s] %(levelname)s  %(message)s",
    datefmt="%H:%M:%S",
)

load_dotenv(find_dotenv())

# api keys
VLLM_API_KEY = os.getenv("VLLM_API_KEY")
OPENROUTER_API_KEY = os.getenv("OPENROUTER_API_KEY")

# model selection
LLM_MODEL_NAME = os.getenv("LLM_MODEL_NAME")
SLM_MODEL_NAME = os.getenv("SLM_MODEL_NAME")
JUDGE_MODEL_NAME = os.getenv("JUDGE_MODEL_NAME")
GENERATOR_MODEL_NAME = os.getenv("GENERATOR_MODEL_NAME")

In [2]:
# dataset generation example

RPS = 5

# instntiate the generator
declarative_generator = AsyncDeclarativeDatasetGenerator(
    # setting up client, prompts and parsers
    client=ChatOpenAI(
        api_key=OPENROUTER_API_KEY,
        base_url="https://openrouter.ai/api/v1",
        model=GENERATOR_MODEL_NAME,
        max_tokens=4092,
        temperature=0.0,
        rate_limiter=InMemoryRateLimiter(
            requests_per_second=RPS,
            check_every_n_seconds=0.1,
            max_bucket_size=RPS * 2
        )
    ),
    # configuring dataset structure (domains, task-types, amount of examples per batch)
    declaration=DatasetDeclaration(
        tasks=["reranking", "context_compression"],
        domains=["coding", "history", "math", "medicine", "research"],
        difficulties=["easy", "medium", "complex"],
        batch_size=10
    ),
    messages_builder=MessagesBuilder.from_sequence(
        ("reranking", PROMPT_REGISTRY.reranking, SyntheticRowReranking),
        ("context_compression", PROMPT_REGISTRY.context_compression, SyntheticRowCompression),
    ),
    rate_limit=RPS
)

In [3]:
rows = await declarative_generator.agenerate_dataset(output_dir="./tmp")

00:47:41 [httpx] INFO  HTTP Request: POST https://openrouter.ai/api/v1/chat/completions "HTTP/1.1 200 OK"
00:47:41 [httpx] INFO  HTTP Request: POST https://openrouter.ai/api/v1/chat/completions "HTTP/1.1 200 OK"
00:47:41 [httpx] INFO  HTTP Request: POST https://openrouter.ai/api/v1/chat/completions "HTTP/1.1 200 OK"
00:47:41 [httpx] INFO  HTTP Request: POST https://openrouter.ai/api/v1/chat/completions "HTTP/1.1 200 OK"
00:47:42 [httpx] INFO  HTTP Request: POST https://openrouter.ai/api/v1/chat/completions "HTTP/1.1 200 OK"
00:47:42 [httpx] INFO  HTTP Request: POST https://openrouter.ai/api/v1/chat/completions "HTTP/1.1 200 OK"
00:47:42 [httpx] INFO  HTTP Request: POST https://openrouter.ai/api/v1/chat/completions "HTTP/1.1 200 OK"
00:47:42 [httpx] INFO  HTTP Request: POST https://openrouter.ai/api/v1/chat/completions "HTTP/1.1 200 OK"
00:47:42 [httpx] INFO  HTTP Request: POST https://openrouter.ai/api/v1/chat/completions "HTTP/1.1 200 OK"
00:47:43 [httpx] INFO  HTTP Request: POST http

In [ ]:
# experimental part


limiter = InMemoryRateLimiter(requests_per_second=3)

# loading data to the BaseDataset subclass
dataset = SlmFlowSyntheticDataset.from_files("generated/slm_flow_df")

# creating routing policies to each task (policies might be different)
policies_map = {
    # using statisical policies for example
    "reranking": ThresholdRoutingPolicy(
        rules={
            "query_noun_chunk_count": ("gt", 4.0,   0.4),
            "avg_lexical_overlap": ("lt", 0.3,   0.5),
            "inter_document_similarity": ("gt", 0.85,  0.5),
            "query_avg_word_frequency": ("lt", 1e-3,  0.4),
        },
        total_threshold=0.8,
        min_triggers=2,
    ),
    "context_compression": ThresholdRoutingPolicy(
        rules={
            "query_noun_chunk_count": ("gt", 4.0,   0.4),
            "avg_lexical_overlap": ("lt", 0.3,   0.5),
            "inter_document_similarity": ("gt", 0.85,  0.5),
            "query_avg_word_frequency": ("lt", 1e-3,  0.4),
        },
        total_threshold=0.8,
        min_triggers=2,
    )
}

# configuring the BaseFeatureExtractor subclasses for each task
extractors_map = {
    "reranking": RerankingFeatureExtractor(),
    "context_compression": ContextCompressionFeatureExtractor(),
}

# Some boilerplate code with api clients

common_params = {
    # for example, it might be openrouter
    "api_key": OPENROUTER_API_KEY,
    "base_url": "https://openrouter.ai/api/v1",
    "max_tokens": 4092,
    "temperature": 0.0,
    "rate_limiter": limiter
}
slm_client = ChatOpenAI(
    model=SLM_MODEL_NAME,
    **common_params
)
llm_client = ChatOpenAI(
    model=LLM_MODEL_NAME,
    **common_params
)
judge_llm = ChatOpenAI(
    model=JUDGE_MODEL_NAME,
    **common_params
)

# setting up scheduler, passing the policies and some language models clients to it
scheduler_dynamic = SlmFlowScheduler(
    policies=policies_map,
    extractors=extractors_map,
    llm=llm_client,
    slm=slm_client
)

# for ablation we can try to perform "clear-model" experiments
scheduler_slm = SlmFlowScheduler({}, extractors_map, llm_client, slm_client, mode="slm_only")
scheduler_llm = SlmFlowScheduler({}, extractors_map, llm_client, slm_client, mode="llm_only")

In [ ]:
# define the Flow models

dynamic_flow = InferenceFlow(
    dataset=dataset[:30], # in example way, just define only 30 rows
    scheduler=scheduler_dynamic,
)
slm_flow = InferenceFlow(
    dataset=dataset[:30],
    scheduler=scheduler_slm,
)
llm_flow = InferenceFlow(
    dataset=dataset[:30],
    scheduler=scheduler_llm,
)

# and routine objects, like parsers, prompt templates and task-abstracts
tasks = {
    "reranking": RagTask,
    "context_compression": RagTask
}
parsers = {
    "reranking": PydanticOutputParser(pydantic_object=RagTaskOutput),
    "context_compression": PydanticOutputParser(pydantic_object=RagTaskOutput)
}
templates = {
    "reranking": PROMPT_REGISTRY.reranking_inference,
    "context_compression": PROMPT_REGISTRY.context_compression_inference,
}
task_scoring_map = {
    "reranking": get_reranking_metrics,
    "context_compression": get_context_compression_metrics
}

In [ ]:
# now, we can run the experiment

# runtime: qwen-3-235b-a22b + ministral-3b-2512 - 240.85
dynamic_predictions = dynamic_flow.execute(
    tasks=tasks,
    parsers=parsers,
    templates=templates
)

In [ ]:
# evaluate the results
evaluation_results_dynamic = await dynamic_flow.evaluate_by_judge(judge_llm, PROMPT_REGISTRY.evaluation)
dump = pd.DataFrame([row.model_dump() for row in evaluation_results_dynamic])
dump.to_csv("generated/dynamic_results_small.csv", index=False)

In [ ]:
# and compute some metrics

# {'bert_score_f1': 0.6601903736591339, 'judge_scores': 10.0}
# {'bert_score_f1': 0.0013928211161068507, 'rouge_2_score': 0.039519637363712994, 'judge_scores': 9.857142857142858}

r_dynamic = compute_stat_metrics(evaluation_results_dynamic, task_scoring_map)
routing_metrics = compute_routing_metrics(evaluation_results_dynamic, "ministral-3b-2512")
print(r_dynamic, routing_metrics, sep="\n")

In [ ]:
# runtime: ministral-3b-2512 - 32.74
slm_only_predictions = slm_flow.execute(
    tasks=tasks,
    parsers=parsers,
    templates=templates
)

In [ ]:
evaluation_results_slm_only = await slm_flow.evaluate_by_judge(judge_llm, PROMPT_REGISTRY.evaluation)
dump = pd.DataFrame([row.model_dump() for row in evaluation_results_slm_only])
dump.to_csv("generated/slm_results_small.csv", index=False)

In [ ]:
# reranking {'bert_score_f1': 0.5748026371002197, 'judge_scores': 9.6}
# compression {'bert_score_f1': -0.0494559415133803, 'rouge_2_score': 0.027890577649053064, 'judge_scores': 9.857142857142858}

r_slm = compute_stat_metrics(evaluation_results_slm_only, task_scoring_map)
print(r_slm)

In [ ]:
# runtime: qwen-3-235b-a22b - 660.32
llm_only_predictions = llm_flow.execute(
    tasks=tasks,
    parsers=parsers,
    templates=templates
)

In [ ]:
evaluation_results_llm_only = await llm_flow.evaluate_by_judge(judge_llm, PROMPT_REGISTRY.evaluation)
dump = pd.DataFrame([row.model_dump() for row in evaluation_results_dynamic])
dump.to_csv("generated/llm_results_small.csv", index=False)

In [ ]:
# reranking {'bert_score_f1': 0.9047325452168783, 'judge_scores': 10.0}
# compression {'bert_score_f1': 0.10744992962905339, 'rouge_2_score': 0.032957120535188, 'judge_scores': 10.0}

r_llm = compute_stat_metrics(evaluation_results_llm_only, task_scoring_map)
print(r_llm)

In [8]:
def foo(key: str, **kwargs):
    s = "{key}, {a}, {b}"
    print(s.format(key=key, **kwargs))


foo("key_1", a=1, b=2)

key_1, 1, 2


In [9]:
import json

file = "generated/slm_flow_df/context_compression/coding/complex/3b5c349ed8d14866ae3dcfe99a94e1d2.json"

with open(file, mode="r", encoding="utf-8") as f:
    content = json.load(f)

In [10]:
content

{'usage_metadata': {'input_tokens': 792,
  'output_tokens': 2440,
  'total_tokens': 3232,
  'input_token_details': {'audio': 0, 'cache_read': 0},
  'output_token_details': {'audio': 0, 'reasoning': 1792}},
 'task_row_model': {'query': 'Given k sorted input files totaling N elements on disk, how should I implement a stable external k-way merge that minimizes disk I/O while using bounded memory? Describe the in-memory data structure, I/O buffering strategy, how to preserve stability on equal keys, and the resulting time and memory complexity.',
  'documents': [{'idx': 1,
    'content': 'Academic description: A standard in-memory approach for k-way merging is to maintain a binary min-heap containing the current head element from each of the k sorted inputs. Each heap entry is a tuple (key, file_id, sequence_pos) so that comparisons use key first and file_id or sequence_pos as a tie-breaker to preserve stability across equal keys. On each output step extract-min and replace it with the nex

In [ ]:
import spacy

text = (
    "Apple Inc. was founded by Steve Jobs in Cupertino, California. "
    "The company has been revolutionizing technology since 1976. "
    "Their latest products include the iPhone 15 and MacBook Pro."
)

nlp = spacy.load("en_core_web_lg")

doc = nlp(text)

t = [t for t in doc]

spacy.tokens.token.Token